# ASTR 223 Final Project: GP Projections

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pybaseball.statcast as pyb
from pybaseball import playerid_reverse_lookup, fielding_stats, batting_stats
from pathlib import Path
import dask.dataframe as dd
import re
import unicodedata

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
# all of the statcast regular season data from 2018-2025 (~1 min)
data = dd.read_parquet('files/*.parquet')
statcast_data = data.query('game_type == "R"').reset_index(drop=True)
statcast_dask_data = statcast_data.compute()

In [3]:
# (~ 30 sec)
def normalize_name(s):
    """
    normalize names: lowercase, remove spaces after periods, strip accents
    """
    s = s.lower()
    s = re.sub(r"\.\s*", ".", s)
    s = unicodedata.normalize('NFKD', s)
    return ''.join(c for c in s if not unicodedata.combining(c))

# finds the total number of BIPs each batter has had in each season
batter_bips = (statcast_dask_data
               .groupby(['batter', 'game_year'])['description']
               .apply(lambda s: (s == 'hit_into_play').sum())
               ).reset_index(name='events').copy()

# finds the positions of all players
years = np.arange(2018, 2026)
position_dfs = []
for yr in years:
    df = fielding_stats(yr, qual=1, split_seasons=True)
    position_dfs.append(df)
position_df = pd.concat(position_dfs, ignore_index=True)
position_df['Name'] = position_df['Name'].apply(normalize_name)

# finds each player's primary position in each season (where they played the most games)
position_df = (position_df.groupby(['IDfg', 'Name', 'Season'], as_index=False)
               .apply(lambda x: x.loc[x['G'].idxmax(), 'Pos'])
               .rename(columns={None: 'primary_pos'})
               )

# the ids of each player + normalizing the name col
player_ids = playerid_reverse_lookup(batter_bips['batter'].unique().tolist(), key_type='mlbam').copy()
player_ids['name'] = player_ids['name_first'] + ' ' + player_ids['name_last']
player_ids['name'] = player_ids['name'].apply(normalize_name)
# normalizes the name for a few edge cases
player_ids['name'] = player_ids['name'].apply(lambda x: 'victor mesa jr.' if x=='victor mesa' 
                                              else 'robert hassell iii' if x == 'robert hassell' 
                                              else 'dashawn keirsey jr.' if x == 'dashawn keirsey' 
                                              else x)

In [4]:
# want to merge the position to the player --> need to figure out id mappings

# all of the ids from fangraphs
fangraph_ids = (position_df[['Name', 'IDfg']]
                .groupby(['Name', 'IDfg'])
                .first()
                .reset_index()
                )
# merged the mlbam ids with the fangraphs ids
all_ids = (player_ids
           .merge(fangraph_ids, left_on=['key_fangraphs'], right_on=['IDfg'], how='left')
           )
# these are the rookies for the most part that don't have fangraphs ids --> found them by their name rather than their id
null_ids = (all_ids[all_ids['IDfg'].isna()]
            .drop(columns=['Name', 'IDfg'])
            .merge(fangraph_ids, left_on=['name'], right_on=['Name'], how='left')
            .reset_index(drop=True)
            )
# keeping the names that weren't rookies
all_ids = (all_ids
           .dropna(subset=['Name', 'IDfg'])
           .reset_index(drop=True)
           ).copy()
# recombined the rookies and the non-rookies
all_player_ids = (pd.concat([all_ids, null_ids])[['name', 'Name', 'key_mlbam', 'key_fangraphs', 'IDfg']]
                  .reset_index(drop=True)
                  ).copy()
# dropping the remaining players because for the most part, these guys are no longer in the league (some retired, different leagues, etc.)
all_player_ids = all_player_ids[all_player_ids['IDfg'].notna()].reset_index(drop=True).copy()

In [5]:
# this finds the primary pos for each player in each season

cols_to_keep = ['IDfg', 'Season', 'primary_pos']
player_primary_pos = batter_bips.merge(all_player_ids, left_on='batter', right_on='key_mlbam', how='left').copy()
player_primary_pos = player_primary_pos[player_primary_pos['name'].notna()].copy() # gets rid of the players i got rid of in all_player_ids
player_primary_pos = player_primary_pos.merge(position_df[cols_to_keep], left_on=['IDfg', 'game_year'], right_on=['IDfg', 'Season'], how='left').copy() # adds primary pos
# deals with the full-time DHs
player_primary_pos['Season'] = player_primary_pos['Season'].fillna(player_primary_pos['game_year'])
player_primary_pos['primary_pos'] = player_primary_pos['primary_pos'].fillna('DH')
player_primary_pos['IDfg'] = player_primary_pos['IDfg'].astype(int)
# deals with ohtani
player_primary_pos.loc[(player_primary_pos['name'] == 'shohei ohtani'), 'primary_pos'] = 'DH'
# filters it to only position players (no pitchers)
player_primary_pos = player_primary_pos[player_primary_pos['primary_pos'] != 'P'].reset_index(drop=True).copy()
player_primary_pos = player_primary_pos.sort_values(by=['name', 'game_year'], ascending=[True, True]).reset_index(drop=True).copy()

In [6]:
# adding age to each season for each player (~1 min)

player_ages = batting_stats(2018, 2025, qual=1)[['IDfg', 'Season', 'Age']].copy()
player_primary_pos = player_primary_pos.merge(player_ages, left_on=['IDfg', 'game_year'], right_on=['IDfg', 'Season'], how='left').copy()
player_primary_pos = player_primary_pos[['game_year', 'batter', 'IDfg', 'name', 'Age', 'primary_pos', 'events']].copy()

In [9]:
batted_ball_cols = ['game_date', 'batter', 'events', 'stand', 'home_team', 'away_team', 'bb_type', 'launch_speed', 'launch_angle']
batted_ball_events = statcast_dask_data[((~statcast_dask_data['launch_speed'].isna()) 
                                         & (~statcast_dask_data['launch_angle'].isna())
                                         & (statcast_dask_data['description'] == 'hit_into_play'))].reset_index(drop=True)
batted_ball_events = batted_ball_events[batted_ball_cols]
batted_ball_events['is_HR'] = (batted_ball_events['events'] == 'home_run').astype(int)
batted_ball_events

,game_date,batter,events,stand,home_team,away_team,bb_type,launch_speed,launch_angle,is_HR
0,2018-10-01,471865,field_out,L,LAD,COL,ground_ball,79.9,-33,0
1,2018-10-01,596115,home_run,R,LAD,COL,fly_ball,104.7,24,1
2,2018-10-01,571448,home_run,R,LAD,COL,fly_ball,107.1,23,1
3,2018-10-01,571771,field_out,R,LAD,COL,ground_ball,81.0,-11,0
4,2018-10-01,624577,field_out,R,LAD,COL,ground_ball,65.7,-16,0
...,...,...,...,...,...,...,...,...,...,...
909229,2025-03-18,683737,field_out,L,CHC,LAD,line_drive,88.9,24,0
909230,2025-03-18,663656,field_out,L,CHC,LAD,ground_ball,60.7,-62,0
909231,2025-03-18,673548,field_out,R,CHC,LAD,line_drive,55.0,31,0
909232,2025-03-18,669242,field_out,R,CHC,LAD,popup,72.5,72,0


In [15]:
player_primary_pos

,game_year,batter,IDfg,name,Age,primary_pos,events
0,2018,454560,5677,a.j.ellis,37,C,119
1,2018,607223,16246,a.j.reed,25,1B,2
2,2019,607223,16246,a.j.reed,26,1B,24
3,2018,571437,11270,aaron altherr,27,RF,154
4,2019,571437,11270,aaron altherr,28,CF,37
...,...,...,...,...,...,...,...
5138,2021,670097,19562,zack short,26,SS,103
5139,2022,670097,19562,zack short,27,SS,6
5140,2023,670097,19562,zack short,28,2B,159
5141,2024,670097,19562,zack short,29,3B,49


In [19]:
batted_ball_events

,game_date,batter,events,stand,home_team,away_team,bb_type,launch_speed,launch_angle,is_HR
0,2018-10-01,471865,field_out,L,LAD,COL,ground_ball,79.9,-33,0
1,2018-10-01,596115,home_run,R,LAD,COL,fly_ball,104.7,24,1
2,2018-10-01,571448,home_run,R,LAD,COL,fly_ball,107.1,23,1
3,2018-10-01,571771,field_out,R,LAD,COL,ground_ball,81.0,-11,0
4,2018-10-01,624577,field_out,R,LAD,COL,ground_ball,65.7,-16,0
...,...,...,...,...,...,...,...,...,...,...
909229,2025-03-18,683737,field_out,L,CHC,LAD,line_drive,88.9,24,0
909230,2025-03-18,663656,field_out,L,CHC,LAD,ground_ball,60.7,-62,0
909231,2025-03-18,673548,field_out,R,CHC,LAD,line_drive,55.0,31,0
909232,2025-03-18,669242,field_out,R,CHC,LAD,popup,72.5,72,0
